# Parquet Pivot Analysis

This notebook is the central orchestration layer for tariff parquet analysis.
Helper modules in the same folder handle path resolution, parquet loading, pivots,
and annual contract cost calculations for the new storage schema.


In [19]:
from pathlib import Path
import importlib
import sys
import pandas as pd

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 180)

NOTEBOOK_DIR = Path.cwd().resolve()
if not (NOTEBOOK_DIR / 'storage_paths.py').exists():
    candidate = Path('Attempt3/Tariff_preprocesssing/new_organization_tariff_data/contract cost calculations').resolve()
    if (candidate / 'storage_paths.py').exists():
        NOTEBOOK_DIR = candidate

if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

importlib.invalidate_caches()
import storage_paths
importlib.reload(storage_paths)
resolve_storage_dir = getattr(storage_paths, 'resolve_storage_dir', None)
if resolve_storage_dir is None:
    resolve_storage_dir = storage_paths.resolve_storage_output_dir
resolve_profile_path = storage_paths.resolve_profile_path
from parquet_loader import load_storage_frames, build_contract_catalog
from tariff_pivots import build_usage_analysis_table, build_fee_analysis_table, provider_year_overview
from annual_contract_costs import calculate_annual_contract_costs

storage_dir = resolve_storage_dir()
profile_path = resolve_profile_path()
print(f'Storage dir: {storage_dir}')
print(f'Profile path: {profile_path}')


Storage dir: C:\Users\20203525\Documents\2025 2026\WB4U\wb4y-webscraper\Attempt3\Tariff_preprocesssing\new_organization_tariff_data\storage files output
Profile path: C:\Users\20203525\Documents\2025 2026\WB4U\wb4y-webscraper\Attempt3\Tariff_preprocesssing\new_organization_tariff_data\contract cost calculations\4c6a58e2-f666-4d3d-b8c0-55d32a781314.json


In [20]:
frames = load_storage_frames(storage_dir)
frame_summary = pd.DataFrame([
    {'name': 'contracts_fixed', 'rows': len(frames.contracts_fixed), 'columns': len(frames.contracts_fixed.columns)},
    {'name': 'contracts_variable', 'rows': len(frames.contracts_variable), 'columns': len(frames.contracts_variable.columns)},
    {'name': 'fixed_usage', 'rows': len(frames.fixed_usage), 'columns': len(frames.fixed_usage.columns)},
    {'name': 'variable_usage', 'rows': len(frames.variable_usage), 'columns': len(frames.variable_usage.columns)},
    {'name': 'fixed_fees', 'rows': len(frames.fixed_fees), 'columns': len(frames.fixed_fees.columns)},
    {'name': 'variable_fees', 'rows': len(frames.variable_fees), 'columns': len(frames.variable_fees.columns)},
])
print(frame_summary.to_string(index=False))


              name  rows  columns
   contracts_fixed  3649       24
contracts_variable  2520       24
       fixed_usage  8978       11
    variable_usage  6228       11
        fixed_fees  7298       10
     variable_fees  5040       10


In [21]:
contract_catalog = build_contract_catalog(frames)
print(f'Contract catalog rows: {len(contract_catalog):,}')
print(f'Providers: {contract_catalog["provider_name"].nunique():,}')
print(f'Offer keys: {contract_catalog["offer_key"].nunique():,}')
print()
print(contract_catalog[[
    'provider_name',
    'contract_base_name',
    'contract_duration_label',
    'contract_type',
    'meter_type',
    'snapshot_month',
    'contract_snapshot_key',
    'contract_base_key',
    'offer_key',
]].head(12).to_string(index=False))


Contract catalog rows: 6,169
Providers: 47
Offer keys: 440

provider_name contract_base_name contract_duration_label contract_type meter_type snapshot_month                                                                  contract_snapshot_key     contract_base_key                                                offer_key
    AllureNRG        Vaste Prijs           Vast (3 jaar)         fixed     double        2025-04 allurenrg|fixed|double|vaste_prijs_3_jaar_zonder_zonnepanelen|vast_3_jaar|none|2025-04 allurenrg|vaste_prijs allurenrg|fixed|allurenrg|vaste_prijs|vast_3_jaar|double
    AllureNRG        Vaste Prijs           Vast (3 jaar)         fixed     single        2025-04    allurenrg|fixed|single|vaste_prijs_3_jaar_met_zonnepanelen|vast_3_jaar|none|2025-04 allurenrg|vaste_prijs allurenrg|fixed|allurenrg|vaste_prijs|vast_3_jaar|single
    AllureNRG        Vaste Prijs           Vast (2 jaar)         fixed     double        2025-04    allurenrg|fixed|double|vaste_prijs_2_jaar_met_zonne

In [22]:
usage_table = build_usage_analysis_table(frames)
fee_table = build_fee_analysis_table(frames)
overview = provider_year_overview(frames, provider_names=['Greenchoice', 'Essent', 'Vattenfall'], year=2025)

print('Usage table sample:')
print(usage_table[[
    'provider_name', 'contract_type', 'commodity', 'tariff_band', 'snapshot_month', 'rate', 'offer_key'
]].head(10).round(6).to_string(index=False))
print()
print('Fee table sample:')
print(fee_table[[
    'provider_name', 'contract_type', 'fee_component', 'snapshot_month', 'annual_amount', 'offer_key'
]].head(10).round(2).to_string(index=False))
print()
print('Provider-year contract overview:')
print(overview['contracts'][[
    'provider_name', 'contract_type', 'contract_base_name', 'contract_duration_label', 'meter_type', 'snapshot_month'
]].head(15).to_string(index=False))
print()
print('Usage pivot sample:')
print(overview['usage_pivot'].head(10).round(6).to_string())
print()
print('Fee pivot sample:')
print(overview['fee_pivot'].head(10).round(2).to_string())


Usage table sample:
provider_name contract_type   commodity tariff_band snapshot_month   rate                                                offer_key
    AllureNRG         fixed electricity        peak        2025-04 0.2844 allurenrg|fixed|allurenrg|vaste_prijs|vast_3_jaar|double
    AllureNRG         fixed electricity     offpeak        2025-04 0.2844 allurenrg|fixed|allurenrg|vaste_prijs|vast_3_jaar|double
    AllureNRG         fixed         gas      single        2025-04 1.2985 allurenrg|fixed|allurenrg|vaste_prijs|vast_3_jaar|double
    AllureNRG         fixed electricity      single        2025-04 0.2844 allurenrg|fixed|allurenrg|vaste_prijs|vast_3_jaar|single
    AllureNRG         fixed         gas      single        2025-04 1.2985 allurenrg|fixed|allurenrg|vaste_prijs|vast_3_jaar|single
    AllureNRG         fixed electricity        peak        2025-04 0.2892 allurenrg|fixed|allurenrg|vaste_prijs|vast_2_jaar|double
    AllureNRG         fixed electricity     offpeak        2025

In [23]:
annual_results = calculate_annual_contract_costs(
    storage_dir=storage_dir,
    profile_path=profile_path,
    annual_gas_m3=800.0,
    year=2025,
)

print('Consumption summary used for annual costs:')
print(annual_results['consumption_summary'].round(3).to_string(index=False))
print()
print('Annual 2025 cost summary by contract type:')
print(annual_results['summary_by_contract_type'].round(2).to_string(index=False))
print()
print('Feed-in cost coverage:')
print(f"  Fixed offers with feed-in cost: {(annual_results['fixed_annual_offer_costs']['feed_in_cost_eur'] > 0).sum()}")
print(f"  Variable offers with feed-in cost: {(annual_results['variable_annual_offer_costs']['feed_in_cost_eur'] > 0).sum()}")
print()
print('Fixed tariff matrix sample:')
print(annual_results['fixed_tariff_matrix'][[
    'provider_name', 'contract_base_name', 'meter_type', 'electricity_peak', 'electricity_offpeak', 'gas'
]].head(10).round(6).to_string(index=False))
print()
print('Fixed usage matrix sample:')
print(annual_results['fixed_usage_matrix'].head(5).round(3).to_string(index=False))
print()
print('Variable monthly tariff matrix sample:')
print(annual_results['variable_monthly_tariff_matrix'].head(12).round(6).to_string(index=False))
print()
print('Variable monthly usage matrix sample:')
print(annual_results['variable_monthly_usage_matrix'].head(12).round(3).to_string(index=False))
print()
print('Variable weighted annual tariff matrix sample:')
print(annual_results['variable_weighted_tariff_matrix'].head(12).round(6).to_string(index=False))


Consumption summary used for annual costs:
 annual_import_kwh  annual_feedin_kwh  annual_net_load_kwh  annual_gas_m3  annual_peak_import_kwh  annual_offpeak_import_kwh
          6500.003           4000.004               2500.0          800.0                3553.622                   2946.381

Annual 2025 cost summary by contract type:
contract_type  offers  providers  avg_total_annual_cost_eur  min_total_annual_cost_eur  max_total_annual_cost_eur
        fixed     227         26                    3785.33                    2684.79                    5339.89
     variable     213         47                    4489.07                    2747.96                   14427.90

Feed-in cost coverage:
  Fixed offers with feed-in cost: 213
  Variable offers with feed-in cost: 200

Fixed tariff matrix sample:
 provider_name                               contract_base_name meter_type  electricity_peak  electricity_offpeak    gas
     AllureNRG                                      Vaste Prijs     

In [24]:
fixed_annual = annual_results['fixed_annual_offer_costs'].copy()
variable_annual = annual_results['variable_annual_offer_costs'].copy()
fixed_display_cols = [
    'provider_name',
    'contract_base_name',
    'contract_duration_label',
    'meter_type',
    'electricity_cost_eur',
    'gas_cost_eur',
    'fixed_fee_cost_eur',
    'feed_in_cost_eur',
    'total_annual_cost_eur',
]
variable_display_cols = [
    'provider_name',
    'contract_base_name',
    'contract_duration_label',
    'meter_type',
    'months_available',
    'electricity_cost_eur',
    'gas_cost_eur',
    'fixed_fee_cost_eur',
    'feed_in_cost_eur',
    'total_annual_cost_eur',
]

print('Fixed annual offer costs for 2025:')
print(fixed_annual[fixed_display_cols].head(20).round(2).to_string(index=False))
print()
print('Fixed offers with non-zero feed-in costs:')
print(fixed_annual[fixed_annual['feed_in_cost_eur'] > 0].sort_values('feed_in_cost_eur', ascending=False)[fixed_display_cols].head(20).round(2).to_string(index=False))
print()
print('Variable annual offer costs for 2025:')
print(variable_annual[variable_display_cols].head(20).round(2).to_string(index=False))
print()
print('Variable offers with non-zero feed-in costs:')
print(variable_annual[variable_annual['feed_in_cost_eur'] > 0].sort_values('feed_in_cost_eur', ascending=False)[variable_display_cols].head(20).round(2).to_string(index=False))


Fixed annual offer costs for 2025:
   provider_name             contract_base_name contract_duration_label meter_type  electricity_cost_eur  gas_cost_eur  fixed_fee_cost_eur  feed_in_cost_eur  total_annual_cost_eur
Coolblue Energie                         Direct          Vast (<1 jaar)     single               1481.35        987.68              215.76              0.00                2684.79
Coolblue Energie                         Direct          Vast (<1 jaar)     double               1481.72        987.68              215.76              0.00                2685.15
Coolblue Energie Coolblue Energie Vast (3 jaar)           Vast (3 jaar)     single               1589.25       1079.20              215.98              0.00                2884.44
Coolblue Energie Coolblue Energie Vast (3 jaar)           Vast (3 jaar)     double               1591.97       1079.20              215.98              0.00                2887.15
Coolblue Energie Coolblue Energie Vast (1 jaar)           Vast (1

## Validate Vattenfall Variable Contract Results

This section rebuilds the Vattenfall 2025 variable-contract results from the monthly tariff, usage, fee, and feed-in inputs so the final annual totals can be checked directly.

In [ ]:
# Rebuild Vattenfall variable-contract totals from the monthly inputs and compare to annual_results
import annual_contract_costs as acc

vattenfall_annual = annual_results['variable_annual_offer_costs'].copy()
vattenfall_annual = vattenfall_annual[
    vattenfall_annual['provider_name'].astype(str).str.lower() == 'vattenfall'
].copy()

if vattenfall_annual.empty:
    print('No Vattenfall variable offers found for 2025.')
else:
    annual_feedin_kwh = float(annual_results['consumption_summary'].iloc[0]['annual_feedin_kwh'])
    month_weights = (
        annual_results['monthly_usage_matrix'][['snapshot_month', 'gas']]
        .assign(month_weight=lambda df: df['gas'] / df['gas'].sum())
        .set_index('snapshot_month')['month_weight']
        .to_dict()
    )
    offer_keys = vattenfall_annual['offer_key'].tolist()

    vf_monthly_tariffs = annual_results['variable_monthly_tariff_matrix'].copy()
    vf_monthly_tariffs = vf_monthly_tariffs[
        vf_monthly_tariffs['offer_key'].isin(offer_keys)
    ].copy()

    vf_monthly_usage = annual_results['variable_monthly_usage_matrix'].copy()
    vf_monthly_usage = vf_monthly_usage[
        vf_monthly_usage['offer_key'].isin(offer_keys)
    ].copy()

    vf_monthly_detail = vf_monthly_tariffs.merge(
        vf_monthly_usage,
        on=['offer_key', 'snapshot_month'],
        how='inner',
        suffixes=('_rate', '_usage'),
    )
    vf_monthly_detail['electricity_peak_cost_eur'] = (
        vf_monthly_detail['electricity_peak_rate'] * vf_monthly_detail['electricity_peak_usage']
    )
    vf_monthly_detail['electricity_offpeak_cost_eur'] = (
        vf_monthly_detail['electricity_offpeak_rate'] * vf_monthly_detail['electricity_offpeak_usage']
    )
    vf_monthly_detail['gas_cost_eur'] = (
        vf_monthly_detail['gas_rate'] * vf_monthly_detail['gas_usage']
    )
    vf_monthly_detail['monthly_energy_cost_eur'] = (
        vf_monthly_detail['electricity_peak_cost_eur']
        + vf_monthly_detail['electricity_offpeak_cost_eur']
        + vf_monthly_detail['gas_cost_eur']
    )

    _, _, variable_fee_rows = acc._prepare_catalog_usage_fees(frames, contract_type='variable', year=2025)
    variable_fee_rows = variable_fee_rows[
        variable_fee_rows['offer_key'].isin(offer_keys)
    ].copy()
    variable_fee_rows['annual_fee_eur'] = pd.to_numeric(
        variable_fee_rows['annual_amount'],
        errors='coerce',
    ).fillna(pd.to_numeric(variable_fee_rows['amount'], errors='coerce'))
    vf_monthly_fee = (
        variable_fee_rows.groupby(['offer_key', 'snapshot_month'], dropna=False)['annual_fee_eur']
        .sum()
        .reset_index()
    )
    vf_monthly_fee['month_weight'] = vf_monthly_fee['snapshot_month'].map(month_weights).fillna(0.0)
    vf_monthly_fee['weighted_fee_eur'] = vf_monthly_fee['annual_fee_eur'] * vf_monthly_fee['month_weight']

    vf_monthly_detail = vf_monthly_detail.merge(
        vf_monthly_fee,
        on=['offer_key', 'snapshot_month'],
        how='left',
    )

    def weighted_avg(group, rate_col, usage_col):
        denominator = group[usage_col].sum()
        if denominator == 0:
            return 0.0
        return (group[rate_col] * group[usage_col]).sum() / denominator

    vf_rate_validation = (
        vf_monthly_detail.groupby('offer_key', dropna=False)
        .apply(
            lambda group: pd.Series({
                'months_recalc': int(group['snapshot_month'].nunique()),
                'electricity_peak_rate_recalc': weighted_avg(group, 'electricity_peak_rate', 'electricity_peak_usage'),
                'electricity_offpeak_rate_recalc': weighted_avg(group, 'electricity_offpeak_rate', 'electricity_offpeak_usage'),
                'gas_rate_recalc': weighted_avg(group, 'gas_rate', 'gas_usage'),
                'electricity_peak_cost_observed_months_eur': group['electricity_peak_cost_eur'].sum(),
                'electricity_offpeak_cost_observed_months_eur': group['electricity_offpeak_cost_eur'].sum(),
                'gas_cost_observed_months_eur': group['gas_cost_eur'].sum(),
                'electricity_cost_observed_months_eur': group['electricity_peak_cost_eur'].sum() + group['electricity_offpeak_cost_eur'].sum(),
            })
        )
        .reset_index()
    )

    vf_fee_validation = (
        vf_monthly_fee.groupby('offer_key', dropna=False)
        .apply(
            lambda group: pd.Series({
                'fee_months_recalc': int(group['snapshot_month'].nunique()),
                'fixed_fee_recalc_eur': (
                    group['weighted_fee_eur'].sum() / group['month_weight'].sum()
                    if group['month_weight'].sum() else 0.0
                ),
            })
        )
        .reset_index()
    )

    vf_validation = vattenfall_annual.merge(vf_rate_validation, on='offer_key', how='left')
    vf_validation = vf_validation.merge(vf_fee_validation, on='offer_key', how='left')
    vf_validation['feed_in_lookup_eur'] = vf_validation['contract_duration_label'].apply(
        lambda duration: float(acc.get_feed_in_tariff('Vattenfall', duration, annual_feedin_kwh) or 0.0)
    )
    vf_validation['electricity_cost_recalc_eur'] = (
        vf_validation['electricity_peak_rate_recalc'] * vf_validation['annual_peak_import_kwh']
        + vf_validation['electricity_offpeak_rate_recalc'] * vf_validation['annual_offpeak_import_kwh']
    )
    vf_validation['gas_cost_recalc_eur'] = (
        vf_validation['gas_rate_recalc'] * vf_validation['annual_gas_m3']
    )
    vf_validation['pre_feed_in_total_reported_eur'] = (
        vf_validation['total_annual_cost_eur'] - vf_validation['feed_in_cost_eur']
    )
    vf_validation['pre_feed_in_total_recalc_eur'] = (
        vf_validation['electricity_cost_recalc_eur']
        + vf_validation['gas_cost_recalc_eur']
        + vf_validation['fixed_fee_recalc_eur']
    )
    vf_validation['total_recalc_eur'] = (
        vf_validation['pre_feed_in_total_recalc_eur'] + vf_validation['feed_in_lookup_eur']
    )
    vf_validation['peak_rate_delta'] = (
        vf_validation['electricity_peak_rate_recalc'] - vf_validation['electricity_peak_rate_eur_per_kwh']
    )
    vf_validation['offpeak_rate_delta'] = (
        vf_validation['electricity_offpeak_rate_recalc'] - vf_validation['electricity_offpeak_rate_eur_per_kwh']
    )
    vf_validation['gas_rate_delta'] = (
        vf_validation['gas_rate_recalc'] - vf_validation['gas_rate_eur_per_m3']
    )
    vf_validation['fee_delta_eur'] = (
        vf_validation['fixed_fee_recalc_eur'] - vf_validation['fixed_fee_cost_eur']
    )
    vf_validation['feed_in_delta_eur'] = (
        vf_validation['feed_in_lookup_eur'] - vf_validation['feed_in_cost_eur']
    )
    vf_validation['pre_feed_in_total_delta_eur'] = (
        vf_validation['pre_feed_in_total_recalc_eur'] - vf_validation['pre_feed_in_total_reported_eur']
    )
    vf_validation['total_delta_eur'] = (
        vf_validation['total_recalc_eur'] - vf_validation['total_annual_cost_eur']
    )

    validation_cols = [
        'contract_base_name',
        'meter_type',
        'months_available',
        'months_recalc',
        'electricity_peak_rate_eur_per_kwh',
        'electricity_peak_rate_recalc',
        'peak_rate_delta',
        'electricity_offpeak_rate_eur_per_kwh',
        'electricity_offpeak_rate_recalc',
        'offpeak_rate_delta',
        'gas_rate_eur_per_m3',
        'gas_rate_recalc',
        'gas_rate_delta',
        'fixed_fee_cost_eur',
        'fixed_fee_recalc_eur',
        'fee_delta_eur',
        'electricity_cost_observed_months_eur',
        'electricity_cost_recalc_eur',
        'gas_cost_observed_months_eur',
        'gas_cost_recalc_eur',
        'feed_in_cost_eur',
        'feed_in_lookup_eur',
        'feed_in_delta_eur',
        'total_annual_cost_eur',
        'total_recalc_eur',
        'total_delta_eur',
        'estimated_annual_costs_source_mean',
    ]
    monthly_cols = [
        'offer_key',
        'snapshot_month',
        'electricity_peak_rate',
        'electricity_peak_usage',
        'electricity_peak_cost_eur',
        'electricity_offpeak_rate',
        'electricity_offpeak_usage',
        'electricity_offpeak_cost_eur',
        'gas_rate',
        'gas_usage',
        'gas_cost_eur',
        'annual_fee_eur',
        'month_weight',
        'monthly_energy_cost_eur',
    ]

    print('Vattenfall variable offers found:', len(vf_validation))
    print('Feed-in lookup source for these rows uses the provider-specific Vattenfall tariff table.')
    print(f"Max abs total delta: {vf_validation['total_delta_eur'].abs().max():.12f} EUR")
    print(f"Max abs fee delta: {vf_validation['fee_delta_eur'].abs().max():.12f} EUR")
    print(f"Max abs rate delta: {vf_validation[['peak_rate_delta', 'offpeak_rate_delta', 'gas_rate_delta']].abs().max().max():.12f}")
    display(vf_validation[validation_cols].sort_values(['total_annual_cost_eur', 'contract_base_name', 'meter_type']).round(6))
    display(vf_monthly_detail[monthly_cols].sort_values(['offer_key', 'snapshot_month']).round(6))


Vattenfall variable offers found: 16
Feed-in lookup source for these rows uses the provider-specific Vattenfall tariff table.
Max abs total delta: 0.000000000001 EUR
Max abs fee delta: 0.000000000000 EUR
Max abs rate delta: 0.000000000000


C:\Users\20203525\AppData\Local\Temp\ipykernel_55724\362706298.py:82: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(
C:\Users\20203525\AppData\Local\Temp\ipykernel_55724\362706298.py:99: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


,contract_base_name,meter_type,months_available,months_recalc,electricity_peak_rate_eur_per_kwh,electricity_peak_rate_recalc,peak_rate_delta,electricity_offpeak_rate_eur_per_kwh,electricity_offpeak_rate_recalc,offpeak_rate_delta,gas_rate_eur_per_m3,gas_rate_recalc,gas_rate_delta,fixed_fee_cost_eur,fixed_fee_recalc_eur,fee_delta_eur,electricity_cost_observed_months_eur,electricity_cost_recalc_eur,gas_cost_observed_months_eur,gas_cost_recalc_eur,feed_in_cost_eur,feed_in_lookup_eur,feed_in_delta_eur,total_annual_cost_eur,total_recalc_eur,total_delta_eur,estimated_annual_costs_source_mean
0,BudgetStroom & BudgetGas,single,8,8.0,0.275712,0.275712,0.0,0.276196,0.276196,0.0,1.386000,1.386000,-0.0,190.378711,190.378711,0.0,1067.721214,1793.553445,738.187178,1108.799671,689.04,689.04,0.0,3781.771827,3781.771827,0.0,169.125000
1,Groen uit Nederland &Gas,single,12,12.0,0.277971,0.277971,0.0,0.278055,0.278055,-0.0,1.368655,1.368655,0.0,194.836521,194.836521,0.0,1807.061063,1807.061063,1094.923836,1094.923836,689.04,689.04,0.0,3785.861419,3785.861419,-0.0,168.500000
2,BudgetStroom & BudgetGas,double,8,8.0,0.285332,0.285332,0.0,0.266836,0.266836,0.0,1.386000,1.386000,-0.0,190.378711,190.378711,0.0,1068.365179,1800.161699,738.187178,1108.799671,689.04,689.04,0.0,3788.380080,3788.380080,0.0,169.250000
3,Stroom &Gas,single,12,12.0,0.279551,0.279551,-0.0,0.279086,0.279086,-0.0,1.363406,1.363406,0.0,194.836521,194.836521,0.0,1815.711597,1815.711597,1090.724822,1090.724822,689.04,689.04,0.0,3790.312939,3790.312939,-0.0,168.166667
4,Stroom &Gas,double,12,12.0,0.283924,0.283924,0.0,0.274647,0.274647,0.0,1.363406,1.363406,0.0,194.836521,194.836521,0.0,1818.173789,1818.173789,1090.724822,1090.724822,689.04,689.04,0.0,3792.775131,3792.775131,0.0,168.000000
5,Groen uit Nederland &Gas,double,12,12.0,0.281576,0.281576,0.0,0.276187,0.276187,0.0,1.372150,1.372150,0.0,194.836521,194.836521,0.0,1814.365768,1814.365768,1097.719671,1097.719671,689.04,689.04,0.0,3795.961960,3795.961960,0.0,169.250000
6,Groen uit Nederland &Gas met CO2-compensatie,double,12,12.0,0.284034,0.284034,0.0,0.278217,0.278217,-0.0,1.378234,1.378234,0.0,197.598901,197.598901,0.0,1829.082658,1829.082658,1102.587178,1102.587178,689.04,689.04,0.0,3818.308737,3818.308737,0.0,170.500000
7,Stroom ActieprijsGas Actieprijs,single,8,8.0,0.280519,0.280519,0.0,0.281007,0.281007,0.0,1.398125,1.398125,-0.0,190.378711,190.378711,0.0,1086.328668,1824.811334,744.645260,1118.500082,689.04,689.04,0.0,3822.730128,3822.730128,0.0,171.625000
8,Groen uit Nederland &Gas met CO2-compensatie,single,12,12.0,0.281236,0.281236,0.0,0.281713,0.281713,0.0,1.383093,1.383093,0.0,200.657250,200.657250,0.0,1829.438641,1829.438641,1106.474521,1106.474521,689.04,689.04,0.0,3825.610412,3825.610412,0.0,171.416667
9,Stroom ActieprijsGas Actieprijs,double,8,8.0,0.290139,0.290139,0.0,0.271647,0.271647,0.0,1.398125,1.398125,-0.0,190.378711,190.378711,0.0,1086.972633,1831.419588,744.645260,1118.500082,689.04,689.04,0.0,3829.338382,3829.338382,0.0,171.625000


,offer_key,snapshot_month,electricity_peak_rate,electricity_peak_usage,electricity_peak_cost_eur,electricity_offpeak_rate,electricity_offpeak_usage,electricity_offpeak_cost_eur,gas_rate,gas_usage,gas_cost_eur,annual_fee_eur,month_weight,monthly_energy_cost_eur
0,vattenfall|variable|vattenfall|budgetstroom_bu...,2025-01,0.2854,539.7143,154.034461,0.2641,292.0678,77.135106,1.3843,67.945205,94.056548,167.7060,0.084932,325.226115
1,vattenfall|variable|vattenfall|budgetstroom_bu...,2025-02,0.2854,356.0341,101.612132,0.2641,261.1098,68.959098,1.3843,61.369863,84.954301,167.7060,0.076712,255.525532
2,vattenfall|variable|vattenfall|budgetstroom_bu...,2025-03,0.2854,278.1980,79.397709,0.2641,277.5158,73.291923,1.3843,67.945205,94.056548,167.7060,0.084932,246.746180
3,vattenfall|variable|vattenfall|budgetstroom_bu...,2025-04,0.2854,194.4788,55.504250,0.2641,228.3320,60.302481,1.4320,65.753425,94.158904,203.7156,0.082192,209.965635
4,vattenfall|variable|vattenfall|budgetstroom_bu...,2025-05,0.2854,149.9304,42.790136,0.2641,214.9061,56.756701,1.4320,67.945205,97.297534,203.7156,0.084932,196.844371
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
171,vattenfall|variable|vattenfall|stroom_gas|vari...,2025-08,0.2753,157.7541,43.429704,0.2753,222.9349,61.373978,1.3143,67.945205,89.300384,203.7156,0.084932,194.104065
172,vattenfall|variable|vattenfall|stroom_gas|vari...,2025-09,0.2831,215.9948,61.148128,0.2831,219.7096,62.199788,1.3184,65.753425,86.689315,203.7156,0.082192,210.037231
173,vattenfall|variable|vattenfall|stroom_gas|vari...,2025-10,0.2831,317.8597,89.986081,0.2831,233.3674,66.066311,1.3184,67.945205,89.578959,203.7156,0.084932,245.631351
174,vattenfall|variable|vattenfall|stroom_gas|vari...,2025-11,0.2773,467.5702,129.657216,0.2773,267.7170,74.237924,1.2996,65.753425,85.453151,203.7156,0.082192,289.348291


: 